## 1️⃣ Environment Setup & GPU Verification

In [ ]:
# ============================================================
# 1.1 Verify GPU is available
# ============================================================
import subprocess

print("=" * 70)
print("🔍 Checking GPU availability...")
print("=" * 70)

# Check NVIDIA GPU
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print("❌ ERROR: No NVIDIA GPU detected!")
    print("Go to Runtime > Change runtime type > Hardware accelerator > GPU")
    raise RuntimeError("GPU not available")

print("\n✅ GPU detected! Proceeding with setup...")

In [ ]:
# ============================================================
# 1.2 Install CUDA-compatible TensorFlow and dependencies
# ============================================================
print("📦 Installing dependencies...")
print("This may take 2-3 minutes...\n")

# Core ML stack (CUDA-compatible versions)
!pip install -q --upgrade pip

# TensorFlow with CUDA support (Colab has CUDA pre-installed)
!pip install -q tensorflow==2.15.0

# Core dependencies matching Mac environment
!pip install -q \
    numpy==1.26.4 \
    pandas==2.2.0 \
    scikit-learn==1.4.0 \
    scipy==1.11.4 \
    joblib==1.3.2 \
    xgboost==2.0.3 \
    matplotlib==3.8.0 \
    plotly==5.18.0 \
    rich==13.7.0 \
    tqdm==4.66.0 \
    PyYAML==6.0.1 \
    python-dotenv==1.0.0 \
    requests==2.31.0 \
    aiohttp==3.9.0 \
    psutil==5.9.0 \
    structlog==24.1.0 \
    pyarrow==15.0.0 \
    h5py==3.10.0

# MLflow for experiment tracking (optional but recommended)
!pip install -q mlflow==2.10.0

print("\n✅ Dependencies installed!")

In [ ]:
# ============================================================
# 1.3 Verify TensorFlow CUDA setup
# ============================================================
import tensorflow as tf

print("=" * 70)
print("🔧 TensorFlow Configuration")
print("=" * 70)
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA available: {tf.test.is_built_with_cuda()}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

# Enable memory growth to prevent OOM
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n✅ Memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠️ Could not set memory growth: {e}")

# Enable mixed precision for faster training
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f"\n✅ Mixed precision enabled: {tf.keras.mixed_precision.global_policy().name}")
print("   (float16 compute, float32 variables - 1.5-2x speedup)")

## 2️⃣ Clone Repository & Setup

In [ ]:
# ============================================================
# 2.1 Clone the ML Engine repository
# ============================================================
import os

REPO_URL = "https://github.com/Raynergy-svg/ml_engine.git"
REPO_DIR = "/content/ml_engine"

# Remove existing directory if it exists
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

print(f"📥 Cloning repository from {REPO_URL}...")
!git clone {REPO_URL} {REPO_DIR}

# Change to repo directory
os.chdir(REPO_DIR)
print(f"\n📂 Working directory: {os.getcwd()}")
print("\n📁 Repository contents:")
!ls -la

In [ ]:
# ============================================================
# 2.2 Create necessary directories
# ============================================================
import os
from pathlib import Path

directories = [
    "trained_data/models",
    "trained_data/checkpoints",
    "trained_data/checkpoints/tensorflow",
    "trained_data/replay/EUR_USD",
    "trained_data/replay/USD_JPY",
    "trained_data/replay/GBP_USD",
    "trained_data/logs",
    "trained_data/scalers",
    "market_data",
]

for d in directories:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"✅ Created: {d}")

print("\n📁 Directory structure ready!")

## 3️⃣ Environment Variables (OANDA API)

In [ ]:
# ============================================================
# 3.1 Set OANDA API credentials
# ============================================================
# ⚠️ IMPORTANT: Replace these with your actual OANDA Practice credentials
# Get them from: https://www.oanda.com/account/tpa/personal_token

import os
from getpass import getpass

print("=" * 70)
print("🔐 OANDA API Configuration")
print("=" * 70)
print("\nEnter your OANDA Practice account credentials.")
print("(These are stored only in this session's memory)\n")

# Option 1: Interactive input (more secure)
USE_INTERACTIVE = True  # Set to False to hardcode (not recommended)

if USE_INTERACTIVE:
    OANDA_API_TOKEN = getpass("OANDA API Token: ")
    OANDA_ACCOUNT_ID = input("OANDA Account ID: ")
else:
    # ⚠️ NOT RECOMMENDED - Only use for testing
    OANDA_API_TOKEN = "YOUR_API_TOKEN_HERE"
    OANDA_ACCOUNT_ID = "YOUR_ACCOUNT_ID_HERE"

# Set environment variables
os.environ["OANDA_API_TOKEN"] = OANDA_API_TOKEN
os.environ["OANDA_ACCOUNT_ID"] = OANDA_ACCOUNT_ID

# Verify (show only last 4 chars)
print(f"\n✅ OANDA_API_TOKEN: ...{OANDA_API_TOKEN[-4:]}")
print(f"✅ OANDA_ACCOUNT_ID: {OANDA_ACCOUNT_ID}")

In [ ]:
# ============================================================
# 3.2 Test OANDA connection
# ============================================================
import sys
sys.path.insert(0, '/content/ml_engine')

try:
    from oanda_practice import OandaPracticeClient
    
    client = OandaPracticeClient.from_env()
    print("✅ OANDA client initialized successfully!")
    print("\n📊 Testing candle fetch...")
    
    # Fetch a small sample to verify connection
    candles = client.fetch_candles(
        instrument="EUR_USD",
        granularity="H1",
        count=10
    )
    print(f"✅ Fetched {len(candles)} candles from OANDA")
    print(f"   Latest: {candles[-1]['time']} Close: {candles[-1]['mid']['c']}")
    
except Exception as e:
    print(f"❌ OANDA connection failed: {e}")
    print("\n⚠️ You can still train using local CSV files.")
    print("   Upload your market data to /content/ml_engine/market_data/")

## 4️⃣ Data Preparation

In [ ]:
# ============================================================
# 4.1 Fetch training data from OANDA
# ============================================================
import pandas as pd
import numpy as np
from datetime import datetime

# Training configuration - IDENTICAL to Mac
INSTRUMENT = "EUR_USD"  # Change as needed: EUR_USD, GBP_USD, USD_JPY
GRANULARITY = "H1"      # H1 = 1 hour candles
CANDLES = 12000         # ~500 trading days

print("=" * 70)
print(f"📊 Fetching {CANDLES} {GRANULARITY} candles for {INSTRUMENT}")
print("=" * 70)

try:
    from oanda_practice import OandaPracticeClient
    
    client = OandaPracticeClient.from_env()
    
    # Fetch candles in batches (OANDA limit is 5000 per request)
    all_candles = []
    remaining = CANDLES
    from_time = None
    
    while remaining > 0:
        batch_size = min(remaining, 5000)
        print(f"  Fetching batch of {batch_size} candles...")
        
        candles = client.fetch_candles(
            instrument=INSTRUMENT.replace("/", "_"),
            granularity=GRANULARITY,
            count=batch_size,
            to_time=from_time
        )
        
        if not candles:
            break
            
        all_candles = candles + all_candles
        from_time = candles[0]['time']
        remaining -= batch_size
        print(f"  Total collected: {len(all_candles)}")
    
    # Convert to DataFrame
    rows = []
    for c in all_candles:
        rows.append({
            'time': c['time'],
            'open': float(c['mid']['o']),
            'high': float(c['mid']['h']),
            'low': float(c['mid']['l']),
            'close': float(c['mid']['c']),
            'volume': int(c['volume']),
        })
    
    df = pd.DataFrame(rows)
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values('time').reset_index(drop=True)
    
    # Save to CSV
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_path = f"market_data/oanda_{INSTRUMENT.replace('/', '_')}_{GRANULARITY}_live_{timestamp}.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"\n✅ Saved {len(df)} candles to {csv_path}")
    print(f"   Date range: {df['time'].min()} to {df['time'].max()}")
    
    DATA_PATH = csv_path
    
except Exception as e:
    print(f"❌ OANDA fetch failed: {e}")
    print("\n📁 Looking for existing CSV files...")
    
    import glob
    csv_files = glob.glob("market_data/*.csv")
    if csv_files:
        print(f"Found: {csv_files}")
        DATA_PATH = csv_files[0]
        print(f"Using: {DATA_PATH}")
    else:
        print("\n⚠️ No data files found. Please upload a CSV to market_data/")
        DATA_PATH = None

In [ ]:
# ============================================================
# 4.2 Data preview and validation
# ============================================================
if DATA_PATH:
    import pandas as pd
    
    df = pd.read_csv(DATA_PATH)
    
    print("=" * 70)
    print("📊 Data Preview")
    print("=" * 70)
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 5 rows:")
    display(df.head())
    print(f"\nLast 5 rows:")
    display(df.tail())
    print(f"\nStatistics:")
    display(df.describe())
    
    # Check for NaN values
    nan_counts = df.isna().sum()
    if nan_counts.any():
        print(f"\n⚠️ NaN values detected:")
        print(nan_counts[nan_counts > 0])
    else:
        print(f"\n✅ No NaN values in data")

## 5️⃣ Training Configuration (CUDA-Optimized)

In [ ]:
# ============================================================
# 5.1 Training hyperparameters - IDENTICAL to Mac (except GPU settings)
# ============================================================

# These settings are identical to the Mac Metal configuration
# Only GPU-specific settings are adapted for CUDA

TRAINING_CONFIG = {
    # === MODEL ARCHITECTURE (IDENTICAL TO MAC) ===
    "model_type": "ensemble",  # Modular ensemble: Transformer + XGBoost + RF + Ridge
    
    # Transformer (Direction Predictor)
    "transformer_d_model": 32,
    "transformer_num_heads": 4,
    "transformer_num_layers": 2,
    "transformer_dff": 64,
    "transformer_dropout": 0.2,
    
    # === TRAINING PARAMETERS (IDENTICAL TO MAC) ===
    "epochs": 200,
    "batch_size": 128,  # Good for both T4 and higher GPUs
    "learning_rate": 0.0003,  # 3e-4, Adam default
    "patience": 20,  # Early stopping patience
    "seq_len": 60,  # Sequence length for Transformer
    
    # === CUDA-SPECIFIC OPTIMIZATIONS ===
    "mixed_precision": True,  # float16 compute (1.5-2x speedup)
    "jit_compile": True,  # XLA compilation (CUDA supports this well)
    "steps_per_execution": 10,  # Reduce Python overhead
    
    # === CONTINUAL LEARNING (IDENTICAL TO MAC) ===
    "use_ema": True,  # Exponential Moving Average
    "ema_decay": 0.999,
    "use_ewc": True,  # Elastic Weight Consolidation
    "ewc_lambda": 1000.0,
    "use_replay_buffer": True,
    "replay_buffer_ratio": 0.10,
    
    # === WALK-FORWARD VALIDATION (IDENTICAL TO MAC) ===
    "cv_folds": 3,  # Walk-forward cross-validation
    "min_train_samples": 4000,
    "test_period": 1000,
    "gap": 24,  # 1 day gap to prevent leakage
    
    # === OVERFITTING PREVENTION (IDENTICAL TO MAC) ===
    "enable_swa": True,  # Stochastic Weight Averaging
    "enable_cosine_restarts": True,
    "overfit_threshold": 0.08,
    "critical_threshold": 0.15,
    "max_acceptable_gap": 0.12,
}

print("=" * 70)
print("⚙️ Training Configuration")
print("=" * 70)
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

## 6️⃣ Run Training

In [ ]:
# ============================================================
# 6.1 Import training modules
# ============================================================
import os
import sys
import logging

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Add repo to path
sys.path.insert(0, '/content/ml_engine')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

print("📦 Importing training modules...")

import numpy as np
import pandas as pd
import tensorflow as tf

from modular_trainers import (
    TrainerConfig,
    TransformerDirectionTrainer,
    XGBoostMomentumTrainer,
    RandomForestRiskTrainer,
    RidgeConfidenceTrainer,
    OverfitPreventionCallback,
)
from modular_data_loaders import (
    compute_normalized_features,
    prepare_direction_data,
    prepare_momentum_data,
    prepare_risk_data,
    prepare_confidence_data,
)
from feature_engineering import FeatureEngineering

print("✅ All modules imported successfully!")

In [ ]:
# ============================================================
# 6.2 Prepare training data
# ============================================================
from rich.console import Console
from rich.panel import Panel

console = Console()

console.print(Panel("[bold blue]Step 1: Data Preparation[/bold blue]"))

# Load data
df = pd.read_csv(DATA_PATH)
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time')

# Rename columns to lowercase
df.columns = [c.lower() for c in df.columns]

console.print(f"  📊 Loaded {len(df)} candles")
console.print(f"  📅 Date range: {df.index.min()} to {df.index.max()}")

# Compute normalized features
console.print("  🔧 Computing normalized features...")
df = compute_normalized_features(df)

# Add technical indicators
fe = FeatureEngineering()
df = fe.add_technical_indicators(df)

# Fill NaN values
df = df.ffill().bfill()

# Drop remaining NaN rows
df = df.dropna()

console.print(f"  ✅ Features computed: {len(df.columns)} columns")
console.print(f"  ✅ Clean rows: {len(df)}")

In [ ]:
# ============================================================
# 6.3 Prepare model-specific datasets
# ============================================================
console.print(Panel("[bold blue]Step 2: Prepare Model-Specific Data[/bold blue]"))

SEQ_LEN = TRAINING_CONFIG["seq_len"]

# Direction data (Transformer)
console.print("  📊 Preparing Direction data (Transformer)...")
direction_data = prepare_direction_data(df, seq_len=SEQ_LEN)
console.print(f"     X_train: {direction_data['X_train'].shape}")
console.print(f"     y_train: {direction_data['y_train'].shape}")

# Momentum data (XGBoost)
console.print("  📊 Preparing Momentum data (XGBoost)...")
momentum_data = prepare_momentum_data(df)
console.print(f"     X_train: {momentum_data['X_train'].shape}")

# Risk data (Random Forest)
console.print("  📊 Preparing Risk data (Random Forest)...")
risk_data = prepare_risk_data(df)
console.print(f"     X_train: {risk_data['X_train'].shape}")

# Confidence data (Ridge)
console.print("  📊 Preparing Confidence data (Ridge)...")
confidence_data = prepare_confidence_data(df)
console.print(f"     X_train: {confidence_data['X_train'].shape}")

console.print("\n✅ All datasets prepared!")

In [ ]:
# ============================================================
# 6.4 Train Transformer (Direction Predictor)
# ============================================================
console.print(Panel("[bold green]Step 3/6: Training Transformer (Direction)[/bold green]"))

# Create trainer config
config = TrainerConfig(
    epochs=TRAINING_CONFIG["epochs"],
    batch_size=TRAINING_CONFIG["batch_size"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    patience=TRAINING_CONFIG["patience"],
    transformer_d_model=TRAINING_CONFIG["transformer_d_model"],
    transformer_num_heads=TRAINING_CONFIG["transformer_num_heads"],
    transformer_num_layers=TRAINING_CONFIG["transformer_num_layers"],
    transformer_dff=TRAINING_CONFIG["transformer_dff"],
    transformer_dropout=TRAINING_CONFIG["transformer_dropout"],
    use_ema=TRAINING_CONFIG["use_ema"],
    ema_decay=TRAINING_CONFIG["ema_decay"],
    use_ewc=TRAINING_CONFIG["use_ewc"],
    ewc_lambda=TRAINING_CONFIG["ewc_lambda"],
)

# Create and train Transformer
transformer_trainer = TransformerDirectionTrainer(config)

console.print(f"  🏗️ Building Transformer model...")
console.print(f"     d_model={config.transformer_d_model}, heads={config.transformer_num_heads}")
console.print(f"     layers={config.transformer_num_layers}, dff={config.transformer_dff}")

# Train with overfitting prevention
transformer_result = transformer_trainer.train(
    X_train=direction_data['X_train'],
    y_train=direction_data['y_train'],
    X_val=direction_data['X_val'],
    y_val=direction_data['y_val'],
    checkpoint_dir="trained_data/checkpoints",
    enable_swa=TRAINING_CONFIG["enable_swa"],
    enable_cosine_restarts=TRAINING_CONFIG["enable_cosine_restarts"],
)

console.print(f"\n✅ Transformer trained!")
console.print(f"   Val Accuracy: {transformer_result.get('val_accuracy', 0):.1%}")
console.print(f"   Balanced Acc: {transformer_result.get('balanced_accuracy', 0):.1%}")

In [ ]:
# ============================================================
# 6.5 Train XGBoost (Momentum Analyzer)
# ============================================================
console.print(Panel("[bold green]Step 4/6: Training XGBoost (Momentum)[/bold green]"))

xgb_trainer = XGBoostMomentumTrainer(config)

xgb_result = xgb_trainer.train(
    X_train=momentum_data['X_train'],
    y_train=momentum_data['y_train'],
    X_val=momentum_data['X_val'],
    y_val=momentum_data['y_val'],
)

console.print(f"\n✅ XGBoost trained!")
console.print(f"   Momentum MAE: {xgb_result.get('momentum_mae', 0):.4f}")
console.print(f"   Accel Accuracy: {xgb_result.get('accel_accuracy', 0):.1%}")

In [ ]:
# ============================================================
# 6.6 Train Random Forest (Risk Assessor)
# ============================================================
console.print(Panel("[bold green]Step 5/6: Training Random Forest (Risk)[/bold green]"))

rf_trainer = RandomForestRiskTrainer(config)

rf_result = rf_trainer.train(
    X_train=risk_data['X_train'],
    y_train=risk_data['y_train'],
    X_val=risk_data['X_val'],
    y_val=risk_data['y_val'],
)

console.print(f"\n✅ Random Forest trained!")
console.print(f"   Drawdown MAE: {rf_result.get('drawdown_mae', 0):.4f}")
console.print(f"   Streak MAE: {rf_result.get('streak_mae', 0):.4f}")

In [ ]:
# ============================================================
# 6.7 Train Ridge (Confidence Scorer)
# ============================================================
console.print(Panel("[bold green]Step 6/6: Training Ridge (Confidence)[/bold green]"))

ridge_trainer = RidgeConfidenceTrainer(config)

ridge_result = ridge_trainer.train(
    X_train=confidence_data['X_train'],
    y_train=confidence_data['y_train'],
    X_val=confidence_data['X_val'],
    y_val=confidence_data['y_val'],
)

console.print(f"\n✅ Ridge trained!")
console.print(f"   Confidence MAE: {ridge_result.get('confidence_mae', 0):.2f}")
console.print(f"   R² Score: {ridge_result.get('r2_score', 0):.3f}")

In [ ]:
# ============================================================
# 6.8 Save all models
# ============================================================
console.print(Panel("[bold blue]Saving Models[/bold blue]"))

import json
from datetime import datetime

MODEL_DIR = "trained_data/models"

# Save Transformer
transformer_trainer.save(f"{MODEL_DIR}/transformer_direction.keras")
console.print(f"  💾 Saved: {MODEL_DIR}/transformer_direction.keras")

# Save XGBoost
xgb_trainer.save(f"{MODEL_DIR}/xgb_momentum.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/xgb_momentum.pkl")

# Save Random Forest
rf_trainer.save(f"{MODEL_DIR}/rf_risk.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/rf_risk.pkl")

# Save Ridge
ridge_trainer.save(f"{MODEL_DIR}/ridge_confidence.pkl")
console.print(f"  💾 Saved: {MODEL_DIR}/ridge_confidence.pkl")

# Save ensemble metadata
metadata = {
    "trained_at": datetime.now().isoformat(),
    "trained_on": "colab_cuda",
    "instrument": INSTRUMENT,
    "granularity": GRANULARITY,
    "candles": CANDLES,
    "config": TRAINING_CONFIG,
    "results": {
        "transformer": transformer_result,
        "xgboost": xgb_result,
        "random_forest": rf_result,
        "ridge": ridge_result,
    }
}

with open(f"{MODEL_DIR}/modular_ensemble.meta.json", 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
console.print(f"  💾 Saved: {MODEL_DIR}/modular_ensemble.meta.json")

console.print("\n✅ All models saved!")

## 7️⃣ Training Summary & Visualization

In [ ]:
# ============================================================
# 7.1 Training summary
# ============================================================
from rich.table import Table

console.print(Panel("[bold green]🎉 Training Complete![/bold green]"))

summary_table = Table(title="Model Performance Summary")
summary_table.add_column("Model", style="cyan")
summary_table.add_column("Metric", style="magenta")
summary_table.add_column("Value", style="green")

summary_table.add_row("Transformer", "Val Accuracy", f"{transformer_result.get('val_accuracy', 0):.1%}")
summary_table.add_row("Transformer", "Balanced Acc", f"{transformer_result.get('balanced_accuracy', 0):.1%}")
summary_table.add_row("XGBoost", "Accel Accuracy", f"{xgb_result.get('accel_accuracy', 0):.1%}")
summary_table.add_row("XGBoost", "Momentum MAE", f"{xgb_result.get('momentum_mae', 0):.4f}")
summary_table.add_row("Random Forest", "Drawdown MAE", f"{rf_result.get('drawdown_mae', 0):.4f}")
summary_table.add_row("Random Forest", "Streak MAE", f"{rf_result.get('streak_mae', 0):.4f}")
summary_table.add_row("Ridge", "R² Score", f"{ridge_result.get('r2_score', 0):.3f}")
summary_table.add_row("Ridge", "Confidence MAE", f"{ridge_result.get('confidence_mae', 0):.2f}")

console.print(summary_table)

## 8️⃣ Download Models

In [ ]:
# ============================================================
# 8.1 Package models for download
# ============================================================
import shutil
from datetime import datetime

# Create zip file with all models
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f"ml_engine_models_{INSTRUMENT.replace('/', '_')}_{timestamp}"

# Create archive
shutil.make_archive(
    f"/content/{zip_name}",
    'zip',
    root_dir='/content/ml_engine',
    base_dir='trained_data/models'
)

print(f"✅ Models packaged: /content/{zip_name}.zip")
print(f"\n📦 Contents:")
!unzip -l /content/{zip_name}.zip | head -20

In [ ]:
# ============================================================
# 8.2 Download to local machine
# ============================================================
from google.colab import files

print("📥 Downloading models to your local machine...")
print("   (This will open a download dialog)\n")

files.download(f"/content/{zip_name}.zip")

print("\n✅ Download started!")
print("\n📋 To use on your Mac:")
print("   1. Unzip the downloaded file")
print("   2. Copy contents to ml_engine/trained_data/models/")
print("   3. Run: buddy analyze --model-type ensemble")

## 9️⃣ Optional: Push to GitHub

In [ ]:
# ============================================================
# 9.1 Commit and push trained models to GitHub
# ============================================================
# ⚠️ Only run this if you want to push models to your repo

PUSH_TO_GITHUB = False  # Set to True to enable

if PUSH_TO_GITHUB:
    from getpass import getpass
    
    print("🔐 GitHub Authentication")
    print("Enter your GitHub Personal Access Token (PAT)")
    print("Create one at: https://github.com/settings/tokens\n")
    
    GITHUB_TOKEN = getpass("GitHub PAT: ")
    GITHUB_USER = input("GitHub Username: ")
    GITHUB_EMAIL = input("GitHub Email: ")
    
    # Configure git
    !git config --global user.name "{GITHUB_USER}"
    !git config --global user.email "{GITHUB_EMAIL}"
    
    # Set remote with token
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Raynergy-svg/ml_engine.git
    
    # Add and commit
    !git add trained_data/models/
    !git commit -m "feat: Add CUDA-trained models from Colab ({INSTRUMENT})"
    
    # Push
    !git push origin main
    
    print("\n✅ Models pushed to GitHub!")
else:
    print("ℹ️ GitHub push disabled. Set PUSH_TO_GITHUB = True to enable.")

---

## 📝 Notes

### GPU Memory Usage
- T4 (16GB): Handles batch_size=128 comfortably
- A100 (40GB): Can increase batch_size to 256-512

### Training Time Estimates
- T4 GPU: ~15-20 minutes for full ensemble
- A100 GPU: ~5-10 minutes

### Differences from Mac (Metal)
- Using CUDA instead of Metal
- XLA compilation enabled (jit_compile=True)
- Same model architecture and hyperparameters

### Troubleshooting
- OOM Error: Reduce batch_size to 64
- OANDA timeout: Fetch smaller batches
- Import errors: Restart runtime and re-run setup cells